# 09 SSL Method Comparison

## Concept
Compare five methods under one shared setup:
- supervised baseline
- self-training
- FixMatch-style
- Mean Teacher
- hybrid (teacher + threshold)


## Why This Method Exists
Single-method notebooks build intuition per algorithm.
This final notebook gives the full picture: which method wins, and why, under the same split.


## Algorithm Intuition
All methods see the same labeled and unlabeled pools.
They differ in how they trust pseudo-labels:
- Self-training: direct model pseudo-labels.
- FixMatch-style: confidence filtering + consistency on strong views.
- Mean Teacher: EMA teacher provides smoother targets.
- Hybrid: EMA teacher plus confidence filtering.


## Before You Run
This notebook defaults to `FAST_DEV_RUN=True` for quick beginner runtime.
Set `FAST_DEV_RUN=False` to run a harder CIFAR-10 few-label comparison.


## Step 1: Imports


In [ ]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt

sys.path.append(str(Path.cwd().parent / 'src'))

from utils.seed import set_seed
from data.mnist import get_mnist_ssl_twoview
from data.cifar10 import get_cifar10_ssl_twoview
from models.small_cnn import SmallCNN
from models.resnet18 import build_resnet18
from methods.supervised import run_supervised
from methods.self_training import run_self_training
from methods.fixmatch import run_fixmatch
from methods.mean_teacher import run_mean_teacher
from methods.hybrid_teacher_threshold import run_hybrid


## Step 2: Shared Configuration


In [ ]:
set_seed(42)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
FAST_DEV_RUN = True

if FAST_DEV_RUN:
    dataset_name = 'MNIST'
    loaders = get_mnist_ssl_twoview('data', labeled_per_class=50, batch_size=128, num_workers=2, seed=42)
    build = SmallCNN
    steps = 8
    tau = 0.95
    tau_start = 0.70
    warmup = 2
    lambda_u = 1.0
    ema_decay = 0.99
else:
    dataset_name = 'CIFAR10'
    loaders = get_cifar10_ssl_twoview('data', labeled_per_class=40, batch_size=128, num_workers=2, seed=42)
    build = build_resnet18
    steps = 20
    tau = 0.95
    tau_start = 0.70
    warmup = 5
    lambda_u = 10.0
    ema_decay = 0.999

config = {
    'seed': 42,
    'dataset': dataset_name,
    'fast_dev_run': FAST_DEV_RUN,
    'steps': steps,
    'tau_final': tau,
    'tau_start': tau_start,
    'warmup_steps': warmup,
    'lambda_u': lambda_u,
    'ema_decay': ema_decay,
}
print(config)
print({'device': str(DEVICE), 'num_labeled': len(loaders.labeled.dataset), 'num_unlabeled': len(loaders.unlabeled.dataset)})


## Step 3: Train Supervised Baseline


In [ ]:
sup_model = build()
sup_opt = torch.optim.SGD(sup_model.parameters(), lr=0.03, momentum=0.9, weight_decay=5e-4)

sup = run_supervised(
    sup_model,
    loaders.labeled,
    loaders.test,
    sup_opt,
    DEVICE,
    epochs=steps,
    use_progress=True,
)


## Step 4: Train Self-Training


In [ ]:
st_model = build()
st_opt = torch.optim.SGD(st_model.parameters(), lr=0.03, momentum=0.9, weight_decay=5e-4)

st = run_self_training(
    st_model,
    loaders.labeled,
    loaders.unlabeled,
    loaders.unlabeled_eval,
    loaders.test,
    st_opt,
    DEVICE,
    rounds=steps,
    threshold=tau,
    threshold_start=tau_start,
    rampup_rounds=warmup,
    use_soft=False,
    max_unlabeled_per_round=10000,
    use_progress=True,
)


## Step 5: Train FixMatch-Style


In [ ]:
fm_model = build()
fm_opt = torch.optim.SGD(fm_model.parameters(), lr=0.03, momentum=0.9, weight_decay=5e-4)

fm = run_fixmatch(
    fm_model,
    loaders.labeled,
    loaders.unlabeled,
    loaders.unlabeled_eval,
    loaders.test,
    fm_opt,
    DEVICE,
    epochs=steps,
    tau=tau,
    lambda_u=lambda_u,
    tau_start=tau_start,
    rampup_epochs=warmup,
    use_progress=True,
)


## Step 6: Train Mean Teacher


In [ ]:
mt_student = build()
mt_teacher = build()
mt_opt = torch.optim.SGD(mt_student.parameters(), lr=0.03, momentum=0.9, weight_decay=5e-4)

mt = run_mean_teacher(
    mt_student,
    mt_teacher,
    loaders.labeled,
    loaders.unlabeled,
    loaders.test,
    mt_opt,
    DEVICE,
    epochs=steps,
    ema_decay=ema_decay,
    lambda_u=lambda_u,
    unlabeled_eval=loaders.unlabeled_eval,
    pseudo_threshold=tau,
    warmup_epochs=warmup,
    use_progress=True,
)


## Step 7: Train Hybrid (Teacher + Threshold)


In [ ]:
hy_student = build()
hy_teacher = build()
hy_opt = torch.optim.SGD(hy_student.parameters(), lr=0.03, momentum=0.9, weight_decay=5e-4)

hy = run_hybrid(
    hy_student,
    hy_teacher,
    loaders.labeled,
    loaders.unlabeled,
    loaders.unlabeled_eval,
    loaders.test,
    hy_opt,
    DEVICE,
    epochs=steps,
    ema_decay=ema_decay,
    tau=tau,
    lambda_u=lambda_u,
    tau_start=tau_start,
    rampup_epochs=warmup,
    use_progress=True,
)


## Step 8: Final Snapshot And Derived Metrics


In [ ]:
histories = {
    'Supervised': sup.history,
    'Self-training': st.history,
    'FixMatch-style': fm.history,
    'Mean Teacher': mt.history,
    'Hybrid': hy.history,
}

rows = []
for method, history in histories.items():
    final = history[-1]
    rows.append(
        {
            'method': method,
            'final_accuracy': float(final.get('val_accuracy', np.nan)),
            'final_pseudo_label_fraction': float(final.get('pseudo_label_fraction', np.nan)),
            'final_pseudo_label_accuracy': float(final.get('pseudo_label_accuracy', np.nan)),
            'final_entropy': float(final.get('entropy', np.nan)),
            'final_ema_gap': float(final.get('ema_gap', np.nan)),
            'final_teacher_disagreement': float(final.get('teacher_student_disagreement', np.nan)),
        }
    )

summary = pd.DataFrame(rows)
baseline_final = float(summary.loc[summary['method'] == 'Supervised', 'final_accuracy'].iloc[0])
summary['delta_vs_supervised'] = summary['final_accuracy'] - baseline_final
summary = summary.sort_values('final_accuracy', ascending=False).reset_index(drop=True)
summary


## Step 9: Diagnostics
### Plot A: Accuracy Comparison
How to read:
- Start with the final ranking.
- Then compare curve shape (smoothness, dips, recovery).


In [ ]:
series = {}
for method, history in histories.items():
    series[method] = {
        'step': np.arange(len(history)),
        'acc': np.array([float(r.get('val_accuracy', np.nan)) for r in history], dtype=float),
        'accept': np.array([float(r.get('pseudo_label_fraction', np.nan)) for r in history], dtype=float),
        'pseudo_acc': np.array([float(r.get('pseudo_label_accuracy', np.nan)) for r in history], dtype=float),
        'entropy': np.array([float(r.get('entropy', np.nan)) for r in history], dtype=float),
    }

plt.figure(figsize=(6.6, 3.8))
for method in ['Supervised', 'Self-training', 'FixMatch-style', 'Mean Teacher', 'Hybrid']:
    curve = series[method]
    style = '--' if method == 'Supervised' else '-'
    plt.plot(curve['step'], curve['acc'], marker='o', linestyle=style, label=method)
plt.title('Validation accuracy comparison')
plt.xlabel('step')
plt.ylabel('accuracy')
plt.ylim(0, 1)
plt.legend(frameon=False, fontsize=8)
plt.tight_layout()


### Plot B: Pseudo-Label Acceptance
How to read:
- Low acceptance means strict filtering or low confidence.
- High acceptance is useful only if pseudo-label quality is also high.


In [ ]:
plt.figure(figsize=(6.6, 3.6))
for method in ['Self-training', 'FixMatch-style', 'Mean Teacher', 'Hybrid']:
    curve = series[method]
    plt.plot(curve['step'], curve['accept'], marker='o', label=method)
plt.title('Pseudo-label acceptance fraction')
plt.xlabel('step')
plt.ylabel('fraction')
plt.ylim(0, 1)
plt.legend(frameon=False, fontsize=8)
plt.tight_layout()


### Plot C: Pseudo-Label Quality And Entropy
How to read:
- Higher pseudo-label accuracy is better.
- Lower entropy means sharper predictions.
- Sharp but wrong predictions are risky, so inspect both plots together.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11.0, 3.5), sharex=True)
for method in ['Self-training', 'FixMatch-style', 'Mean Teacher', 'Hybrid']:
    curve = series[method]
    axes[0].plot(curve['step'], curve['pseudo_acc'], marker='o', label=method)
    axes[1].plot(curve['step'], curve['entropy'], marker='o', label=method)

axes[0].set_title('Pseudo-label accuracy')
axes[0].set_xlabel('step')
axes[0].set_ylabel('accuracy')
axes[0].set_ylim(0, 1)

axes[1].set_title('Prediction entropy')
axes[1].set_xlabel('step')
axes[1].set_ylabel('entropy')

axes[0].legend(frameon=False, fontsize=8)
plt.tight_layout()


### Plot D: Gain Over Supervised Baseline
Positive values mean the SSL method is outperforming supervised at that step.


In [ ]:
sup_curve = series['Supervised']['acc']

plt.figure(figsize=(6.6, 3.4))
plt.axhline(0.0, color='gray', linestyle='--', linewidth=1)
for method in ['Self-training', 'FixMatch-style', 'Mean Teacher', 'Hybrid']:
    method_curve = series[method]['acc']
    n = min(len(sup_curve), len(method_curve))
    gap = method_curve[:n] - sup_curve[:n]
    plt.plot(np.arange(n), gap, marker='o', label=method)
plt.title('Accuracy gain vs supervised baseline')
plt.xlabel('step')
plt.ylabel('delta accuracy')
plt.legend(frameon=False, fontsize=8)
plt.tight_layout()


## Expected Observations And Interpretation
Typical patterns in this setup:
- `tau=0.95` methods often accept fewer pseudo-labels early, but those labels are cleaner.
- Mean Teacher and Hybrid usually have smoother curves than plain self-training.
- FixMatch-style often improves quickly if acceptance is not too low.

If methods finish close together:
- This does not mean SSL is useless.
- It usually means the setup is not strongly label-starved enough, or run length is short.
- Use the diagnostics to explain behavior: acceptance (quantity), pseudo-label accuracy (quality), and entropy (certainty).

Interpretation rule:
judge methods by **final accuracy + training trajectory + pseudo-label signals**, not final accuracy alone.


## Key Takeaways
- This notebook is a decision dashboard, not just a leaderboard.
- Accuracy tells who won this run; acceptance and pseudo-label quality explain why.
- Re-run with different seeds before making strong conclusions.


## Failure Modes
- Very strict thresholds can collapse unlabeled signal (near-zero acceptance).
- Very loose thresholds can amplify pseudo-label noise.
- Short runs can hide long-horizon differences between methods.
